# starling — DFXM analysis walkthrough

Step-by-step interactive workflow for **strain sweep**, **mosaicity**, and **3D strain-mosa** scans.  
Device is auto-detected: CUDA on ESRF GPU nodes → MPS on Apple Silicon → CPU fallback.

**How to use:** Edit the `CONFIG` cell (Section 1), then run top to bottom with *Run All*.

## 0 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import time, os, h5py

import starling
from starling import properties

FWHM = 2 * np.sqrt(2 * np.log(2))   # sigma → FWHM conversion factor

print(f'starling {starling.__version__}  |  device: {starling.get_device(None)}')


def list_scans(dataset_folder):
    """Print every scan command (scan_id ending in .1) found in the dataset folder."""
    folder = Path(dataset_folder)
    h5_files = sorted(folder.glob('*.h5'))
    if not h5_files:
        print(f'No .h5 file found in {folder}'); return
    h5 = h5_files[0]
    print(f'File: {h5.name}\n')
    with h5py.File(h5) as f:
        keys = sorted(
            [k for k in f.keys() if '.' in k and k[0].isdigit() and k.endswith('.1')],
            key=lambda s: int(s.split('.')[0])
        )
        print(f'{"scan_id":>8}  command')
        print('-' * 90)
        for sid in keys:
            try:
                cmd = f[sid]['title'][()].decode()
                # also show frame count
                def _find_data(name, obj):
                    if (isinstance(obj, h5py.Dataset)
                            and obj.ndim == 3 and obj.shape[1] > 100):
                        raise StopIteration(obj.shape[0])
                nf = '?'
                try: f[sid].visititems(_find_data)
                except StopIteration as e: nf = e.args[0]
                print(f'{sid:>8}  [{nf} frames]  {cmd}')
            except Exception:
                pass
    return h5


def make_symmetric_norm(arr, mask, n_sigma=3, fallback=40e-3):
    """Return (vmin, vmax) centred on median of arr[mask], ±n_sigma*std."""
    if mask.any():
        centre = np.nanmedian(arr[mask])
        spread = max(np.nanstd(arr[mask]), fallback)
        return centre - n_sigma * spread, centre + n_sigma * spread
    return np.nanmin(arr), np.nanmax(arr)

## 1 · Choose your scan  ← **EDIT THIS CELL**

Uncomment **one** option block (A, B, or C), set `RAW_DATA` for your system, then run.

In [ ]:
# ─── change this path for the ESRF cluster ────────────────────────────────────
RAW_DATA = '/Volumes/Elements/MA6278/RAW_DATA'

# ═══════════════════════════════════════════════════════════════════════════════
# OPTION A — Strain sweep  (fscan2d ccmth × mu)
# Each scan is one time-point in the charging series.  Edit scan_id to pick
# a different time-point ('2.1', '3.1', … '37.1').
# ═══════════════════════════════════════════════════════════════════════════════
DATASET_NAME = 'DFXM_insitu_repaired_cell_charging/DFXM_insitu_repaired_cell_charging_g1_initial_strain_sweeps_during_charge_good'
SCAN_ID      = '1.1'
SCAN_TYPE    = 'strain_sweep'

# ═══════════════════════════════════════════════════════════════════════════════
# OPTION B — Mosaicity scan  (fscan2d chi × mu)
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET_NAME = 'DFXM_insitu_repaired_cell_charging/DFXM_insitu_repaired_cell_charging_g1_mosa_projection_charging_1'
# SCAN_ID      = '1.1'
# SCAN_TYPE    = 'mosa'

# ═══════════════════════════════════════════════════════════════════════════════
# OPTION C — 3D strain-mosa layer  (chi × mu × obpitch stack)
# Load all 11 obpitch steps as a single 5-D dataset.
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET_NAME = 'DFXM_insitu_repaired_cell_charging/DFXM_insitu_repaired_cell_charging_g1_2x_strain_mosa_layer_charging_2'
# SCAN_ID      = [f'{i}.1' for i in range(1, 12)]   # list → stacked load
# SCAN_TYPE    = 'strain_mosa_3d'

# ─── preprocessing settings (rarely need changing) ───────────────────────────
SIG_THRESHOLD = 50      # counts: pixels below this are masked in result maps
N_BG_FRAMES   = 5       # number of darkest frames averaged for background
HOT_SIGMA     = 5.0     # hot-pixel detection: σ above local 3×3 median
ROI_THRESHOLD = 0.05    # auto_roi: fraction of z-sum max defining the grain box
ROI_PAD       = 20      # pixels of padding around the grain box
CMAP_MDEG     = 40      # ±N mdeg symmetric range for orientation/strain maps

# ─── derived (do not edit) ────────────────────────────────────────────────────
_ds_base = Path(DATASET_NAME).name
H5_PATH  = f'{RAW_DATA}/{DATASET_NAME}/{_ds_base}.h5'
SCAN_MOTOR_3D = 'instrument/positioners/obpitch'

print(f'Dataset : {_ds_base}')
print(f'Scan ID : {SCAN_ID}')
print(f'Type    : {SCAN_TYPE}')
print(f'H5      : {H5_PATH}')
print(f'Exists  : {Path(H5_PATH).exists()}')

## 2 · Browse available scans *(optional)*

Run this cell to see every scan command in your dataset — useful for picking a different `SCAN_ID` in the CONFIG cell.

In [ ]:
list_scans(f'{RAW_DATA}/{DATASET_NAME}')

## 3 · Load scan

Data is read from the BLISS HDF5 master file into RAM.  
Typical times: ~25 s from USB3 drive; much faster from NVMe or network-attached ESRF storage.  
For the 3D option, a progress bar shows the 11-scan stack being assembled.

In [ ]:
t0 = time.perf_counter()

if SCAN_TYPE == 'strain_mosa_3d':
    dset = starling.DataSet(H5_PATH, scan_id=SCAN_ID, scan_motor=SCAN_MOTOR_3D)
else:
    dset = starling.DataSet(H5_PATH, scan_id=SCAN_ID)

print(f'Loaded in {time.perf_counter()-t0:.1f} s')
print()
dset.info()
print()
print(f'data shape  : {dset.data.shape}')
print(f'dtype       : {dset.dtype}')
print(f'device      : {dset.device}')
sz_mb = dset.data.nbytes / 1e6
print(f'RAM in use  : {sz_mb:.0f} MB')

## 4 · Background subtraction

Background is estimated as the pixel-wise mean of the `N_BG_FRAMES` darkest frames — these are frames with the lowest total detector signal, which in a rocking scan are the off-Bragg frames where only readout noise is recorded.  
Subtraction is clamped so `uint16` never wraps.

In [ ]:
bg = dset.estimate_background(n_lowest=N_BG_FRAMES)
print(f'Background — mean: {bg.mean():.1f} cts   max: {int(bg.max())} cts   '
      f'using {N_BG_FRAMES} darkest frames')

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(bg.astype(float), cmap='hot')
ax.set_title('Background image (mean of darkest frames)')
plt.colorbar(im, ax=ax, label='counts')
plt.tight_layout()
plt.show()

dset.subtract(bg)
print('Background subtracted.')

## 5 · ROI selection

The z-sum (detector integrated over all motor steps) reveals where the grain is.  
**Auto-ROI** thresholds at `ROI_THRESHOLD × max(z-sum)` and adds `ROI_PAD` pixels of padding.  

If the auto box misses the grain — uncomment the manual override block below and set your own pixel range.

In [ ]:
# Compute z-sum before cropping so we can preview the full detector
zsum_full = dset.data.reshape(*dset.data.shape[:2], -1).sum(-1).astype(float)
ny_full, nx_full = zsum_full.shape

# Propose the auto-ROI bounding box
roi = starling.preprocess.auto_roi(
    dset.data, threshold_rel=ROI_THRESHOLD, pad=ROI_PAD
)
r1, r2, c1, c2 = roi
pixel_mm = 6.5e-3   # mm per pixel (6.5 µm PCO pixel)
print(f'Auto ROI : rows {r1}–{r2}, cols {c1}–{c2}')
print(f'           {r2-r1} × {c2-c1} px = '
      f'{(r2-r1)*pixel_mm:.2f} × {(c2-c1)*pixel_mm:.2f} mm')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.imshow(zsum_full, cmap='hot', interpolation='nearest')
rect = plt.Rectangle((c1, r1), c2-c1, r2-r1,
                       edgecolor='cyan', facecolor='none', lw=2)
ax.add_patch(rect)
ax.set_title('Full detector z-sum  (cyan = auto ROI)')
ax.set_xlabel('detector column'); ax.set_ylabel('detector row')

ax = axes[1]
ax.imshow(zsum_full[r1:r2, c1:c2], cmap='hot', interpolation='nearest')
ax.set_title(f'Inside auto ROI  ({r2-r1}×{c2-c1} px)')
ax.set_xlabel('detector column'); ax.set_ylabel('detector row')

plt.tight_layout()
plt.show()

# ── Manual override — uncomment and edit if the auto box misses the grain ─────
# roi = (800, 1300, 900, 1500)   # (row_min, row_max, col_min, col_max)
# r1, r2, c1, c2 = roi
# print(f'Using manual ROI: {roi}')

# Apply ROI: crops dset.data in place
r1, r2, c1, c2 = roi
dset.data = np.ascontiguousarray(dset.data[r1:r2, c1:c2])
print(f'Data shape after crop: {dset.data.shape}')

## 6 · Hot-pixel removal *(optional — slow on large frames)*

Replaces isolated hot pixels with their 3×3 spatial median.  Detection threshold: `HOT_SIGMA × MAD` per frame.  
Skip this cell if the dataset is large and you do not see obvious streaks in the grain maps.

In [ ]:
n_frames = int(np.prod(dset.data.shape[2:]))
ny, nx = dset.data.shape[:2]
print(f'Running hot-pixel filter on {n_frames} frames of {ny}×{nx} px '
      f'— expect ~{n_frames*0.15:.0f} s on MPS/CPU ...')
t0 = time.perf_counter()
dset.remove_hot_pixels(n_sigma=HOT_SIGMA)
print(f'Done in {time.perf_counter()-t0:.1f} s')

## 7 · Grain overview

Z-sum after all preprocessing.  The signal mask (pixels above `SIG_THRESHOLD` in any frame) is used throughout to exclude background in result maps and statistics.

In [ ]:
ny, nx = dset.data.shape[:2]
zsum = dset.data.reshape(ny, nx, -1).sum(-1).astype(float)

# Signal mask: pixels where at least one frame exceeds the threshold
sig_mask = dset.data.reshape(ny, nx, -1).max(-1) > SIG_THRESHOLD

extent = [0, nx * pixel_mm, ny * pixel_mm, 0]   # mm

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
im = ax.imshow(zsum, cmap='hot', extent=extent, interpolation='nearest')
ax.set_xlabel('mm'); ax.set_ylabel('mm')
ax.set_title('Grain — z-sum (integrated counts)')
plt.colorbar(im, ax=ax, label='counts')

ax = axes[1]
ax.imshow(sig_mask.astype(float), cmap='Greys_r', extent=extent,
           vmin=0, vmax=1, interpolation='nearest')
ax.set_xlabel('mm'); ax.set_ylabel('mm')
ax.set_title(f'Signal mask  (>{SIG_THRESHOLD} cts in any frame)')

plt.tight_layout()
plt.show()

print(f'Grain pixels : {sig_mask.sum():,} / {ny*nx:,}  '
      f'({100*sig_mask.mean():.1f} % of ROI)')

## 8 · Fitting

The correct fit is selected automatically from `SCAN_TYPE`:

| Type | Method | Output |
|------|--------|--------|
| `strain_sweep` | `fit_1D_gaussian` per µ layer | (ny, nx, 6, n_µ) — [A, σ, ccmth_centre, k, m, ok] |
| `mosa` | `moments` + `fit_2D_gaussian` | moments (ny,nx,2/2×2); gauss (ny,nx,8) |
| `strain_mosa_3d` | `moments` (3D) | (ny, nx, 3) + (ny, nx, 3, 3) |

> **Note:** the first `fit_1D_gaussian` / `fit_2D_gaussian` call per session pays a ~5–10 s `torch.compile` warm-up.  Subsequent calls use the cached kernel and are fast.

In [ ]:
if SCAN_TYPE == 'strain_sweep':
    # data: (ny, nx, n_ccmth, n_mu)
    n_ccmth = dset.data.shape[2]
    n_mu    = dset.data.shape[3]
    mu_values = dset.motors[1, 0, :]   # mu at each layer
    print(f'Strain sweep: {n_ccmth} ccmth steps × {n_mu} µ layers')
    print(f'ccmth range : {dset.motors[0,:,0].min():.5f}° → {dset.motors[0,:,0].max():.5f}°')
    print(f'µ range     : {mu_values.min():.3f}° → {mu_values.max():.3f}°')
    print()
    print('Fitting Gaussian per µ layer (compile warm-up on first layer ~5-10 s)...')
    all_params = []
    for k in range(n_mu):
        t0 = time.perf_counter()
        p = properties.fit_1D_gaussian(
            dset.data[:, :, :, k],    # (ny, nx, n_ccmth)
            dset.motors[:1, :, k],    # (1, n_ccmth)
            device=dset.device
        )
        dt = time.perf_counter() - t0
        ok_k = (p[..., 5] > 0.5) & sig_mask & (p[..., 0] > SIG_THRESHOLD)
        print(f'  µ layer {k:2d}  µ={mu_values[k]:.3f}°  '
              f'success: {ok_k.sum():5,}  ({dt:.2f} s)')
        all_params.append(p)
    all_params = np.stack(all_params, axis=-1)   # (ny, nx, 6, n_mu)
    print(f'\nResult array shape: {all_params.shape}  [ny, nx, params, µ_layer]')

elif SCAN_TYPE == 'mosa':
    print('Computing moments...')
    t0 = time.perf_counter()
    mean_map, cov_map = dset.moments()
    print(f'moments()         : {time.perf_counter()-t0:.2f} s  → '
          f'mean {mean_map.shape}, cov {cov_map.shape}')
    print()
    print('Fitting 2D Gaussian (compile warm-up on first call ~5-10 s)...')
    t0 = time.perf_counter()
    p2d = dset.fit_2D_gaussian()
    ok  = (p2d[..., 7] > 0.5) & sig_mask
    print(f'fit_2D_gaussian() : {time.perf_counter()-t0:.2f} s  → '
          f'success {ok.sum():,} / {sig_mask.sum():,} grain pixels '
          f'({100*ok.sum()/max(sig_mask.sum(),1):.1f} %)')

elif SCAN_TYPE == 'strain_mosa_3d':
    n_chi = dset.data.shape[2]
    n_mu  = dset.data.shape[3]
    n_ob  = dset.data.shape[4]
    ob_values = dset.motors[2, 0, 0, :]
    print(f'3D scan: chi={n_chi} × µ={n_mu} × obpitch={n_ob}')
    print(f'obpitch range : {ob_values.min():.4f}° → {ob_values.max():.4f}°')
    print()
    print('Computing 3D moments...')
    t0 = time.perf_counter()
    mean_3d, cov_3d = dset.moments()
    print(f'moments() : {time.perf_counter()-t0:.2f} s  → '
          f'mean {mean_3d.shape}  [ny, nx, (chi, µ, obpitch)]')

## 9 · Result maps

All maps are masked to `sig_mask` (grain pixels only); background shows as white.  
Orientation / strain maps are displayed as **deviations from the median** in **millidegrees**.

In [ ]:
# helper: nan-mask outside sig_mask
def masked(arr):
    out = arr.astype(float).copy()
    out[~sig_mask] = np.nan
    return out

# helper: deviation map in millidegrees
def dev_mdeg(arr, reference_mask=None):
    m = reference_mask if reference_mask is not None else sig_mask
    centre = np.nanmedian(arr[m])
    return (arr - centre) * 1000


# ─── STRAIN SWEEP ────────────────────────────────────────────────────────────
if SCAN_TYPE == 'strain_sweep':
    # Show up to 6 evenly spaced µ layers as map grid
    n_show  = min(n_mu, 6)
    k_show  = np.round(np.linspace(0, n_mu-1, n_show)).astype(int)

    fig, axes = plt.subplots(3, n_show, figsize=(3.5*n_show, 9),
                              constrained_layout=True)
    if n_show == 1: axes = axes[:, None]

    for col, k in enumerate(k_show):
        p   = all_params[..., k]                              # (ny, nx, 6)
        ok  = (p[..., 5] > 0.5) & sig_mask & (p[..., 0] > SIG_THRESHOLD)

        # Row 0 — peak centre deviation (mdeg)
        d = masked(dev_mdeg(p[..., 2], ok))
        im = axes[0, col].imshow(d, cmap='RdBu_r',
                                  vmin=-CMAP_MDEG, vmax=CMAP_MDEG,
                                  interpolation='nearest')
        axes[0, col].set_title(f'µ={mu_values[k]:.3f}°', fontsize=9)
        plt.colorbar(im, ax=axes[0, col], label='mdeg' if col==n_show-1 else '')
        if col == 0: axes[0, col].set_ylabel('Δccmth (mdeg)')

        # Row 1 — FWHM in degrees
        fw = np.where(ok, np.abs(p[..., 1]) * FWHM, np.nan)
        im = axes[1, col].imshow(fw, cmap='viridis',
                                  vmin=0, vmax=0.02,
                                  interpolation='nearest')
        plt.colorbar(im, ax=axes[1, col], label='°' if col==n_show-1 else '')
        if col == 0: axes[1, col].set_ylabel('FWHM (°)')

        # Row 2 — amplitude
        im = axes[2, col].imshow(np.where(ok, p[..., 0], np.nan),
                                  cmap='hot', vmin=0,
                                  interpolation='nearest')
        plt.colorbar(im, ax=axes[2, col], label='cts' if col==n_show-1 else '')
        if col == 0: axes[2, col].set_ylabel('Amplitude (cts)')

    for ax_row in axes:
        for ax in ax_row: ax.set_xticks([]); ax.set_yticks([])

    fig.suptitle(f'Strain sweep maps — {_ds_base}', y=1.01)
    plt.show()

    # depth profile
    med_centre, med_fwhm = [], []
    for k in range(n_mu):
        p = all_params[..., k]
        ok = (p[..., 5] > 0.5) & sig_mask & (p[..., 0] > SIG_THRESHOLD)
        med_centre.append(np.nanmedian(p[ok, 2]) if ok.any() else np.nan)
        med_fwhm.append(np.nanmedian(np.abs(p[ok, 1]) * FWHM) if ok.any() else np.nan)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(mu_values, med_centre, 'o-', color='steelblue', lw=2, ms=6)
    axes[0].set_xlabel('µ (°)'); axes[0].set_ylabel('Median ccmth peak centre (°)')
    axes[0].set_title('Strain depth profile'); axes[0].grid(True, alpha=0.3)

    axes[1].plot(mu_values, med_fwhm, 's-', color='coral', lw=2, ms=6)
    axes[1].set_xlabel('µ (°)'); axes[1].set_ylabel('Median FWHM (°)')
    axes[1].set_title('Rocking width vs depth'); axes[1].grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()


# ─── MOSAICITY ───────────────────────────────────────────────────────────────
elif SCAN_TYPE == 'mosa':
    ok = (p2d[..., 7] > 0.5) & sig_mask

    chi_dev  = masked(dev_mdeg(p2d[..., 1], ok))
    mu_dev   = masked(dev_mdeg(p2d[..., 2], ok))
    fwhm_chi = np.where(ok, FWHM * np.sqrt(np.maximum(p2d[..., 3], 0)) * 1000, np.nan)
    fwhm_mu  = np.where(ok, FWHM * np.sqrt(np.maximum(p2d[..., 5], 0)) * 1000, np.nan)
    amp      = np.where(ok, p2d[..., 0], np.nan)
    mosa_tot = np.sqrt(np.where(ok, fwhm_chi**2 + fwhm_mu**2, np.nan))

    maps_mosa = [
        (chi_dev,  'RdBu_r', -CMAP_MDEG, CMAP_MDEG, 'Δchi from median (mdeg)'),
        (mu_dev,   'RdBu_r', -CMAP_MDEG, CMAP_MDEG, 'Δµ from median (mdeg)'),
        (amp,      'hot',    0,           None,       'Amplitude (cts)'),
        (fwhm_chi, 'viridis', 0,          None,       'FWHM chi (mdeg)'),
        (fwhm_mu,  'viridis', 0,          None,       'FWHM µ (mdeg)'),
        (mosa_tot, 'plasma',  0,          None,       'Total mosaicity FWHM (mdeg)'),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
    for ax, (data, cmap, vmin, vmax, title) in zip(axes.flat, maps_mosa):
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
        ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, shrink=0.85)
    fig.suptitle(f'Mosaicity maps — {_ds_base}')
    plt.show()


# ─── 3D STRAIN-MOSA ──────────────────────────────────────────────────────────
elif SCAN_TYPE == 'strain_mosa_3d':
    # Intensity depth profile
    I_per_layer  = dset.data.sum(axis=(0, 1, 2, 3))   # (n_ob,)
    peak_layer   = int(np.argmax(I_per_layer))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(ob_values, I_per_layer / I_per_layer.max(),
             'o-', color='steelblue', lw=2, ms=7)
    ax.axvline(ob_values[peak_layer], ls='--', color='crimson', lw=1.5,
                label=f'peak: {ob_values[peak_layer]:.4f}° (layer {peak_layer+1})')
    ax.set_xlabel('obpitch (°)'); ax.set_ylabel('Normalised integrated intensity')
    ax.set_title('Depth profile — intensity vs obpitch layer')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    # Z-sum images at each depth layer
    ncols = min(n_ob, 6)
    nrows = (n_ob + ncols - 1) // ncols
    fig, axes_g = plt.subplots(nrows, ncols,
                                figsize=(3*ncols, 3*nrows),
                                constrained_layout=True)
    axes_flat = np.array(axes_g).flatten()
    for i in range(n_ob):
        lsum = dset.data[:, :, :, :, i].reshape(ny, nx, -1).sum(-1).astype(float)
        axes_flat[i].imshow(lsum, cmap='hot', interpolation='nearest')
        axes_flat[i].set_title(f'ob={ob_values[i]:.4f}°', fontsize=8)
        axes_flat[i].axis('off')
    for j in range(n_ob, len(axes_flat)):
        axes_flat[j].axis('off')
    fig.suptitle('Z-sum at each obpitch depth layer')
    plt.show()

    # 3D COM maps
    labels_3d = ['chi COM', 'µ COM', 'obpitch COM']
    cmaps_3d  = ['RdBu_r', 'RdBu_r', 'plasma']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
    for i in range(3):
        val = np.where(sig_mask, mean_3d[..., i], np.nan)
        centre = np.nanmedian(val[sig_mask])
        spread = max(np.nanstd(val[sig_mask]), 0.001)
        vmin = centre - 3*spread if i < 2 else np.nanmin(val[sig_mask])
        vmax = centre + 3*spread if i < 2 else np.nanmax(val[sig_mask])
        im = axes[i].imshow(val, cmap=cmaps_3d[i], vmin=vmin, vmax=vmax,
                             interpolation='nearest')
        axes[i].set_title(labels_3d[i])
        axes[i].set_xticks([]); axes[i].set_yticks([])
        plt.colorbar(im, ax=axes[i], label='°')
    fig.suptitle(f'3D COM maps — {_ds_base}')
    plt.show()

## 10 · Statistics

Distributions over all grain pixels (masked to `sig_mask`).  Each histogram shows the **spread within a single grain image** — the width reflects real orientation / strain heterogeneity inside the grain.

In [ ]:
print('\n' + '─'*60)

if SCAN_TYPE == 'strain_sweep':
    # Find the µ layer with most successfully fitted grain pixels
    best_k = max(range(n_mu),
                 key=lambda k: ((all_params[...,k][...,5]>0.5)
                                & sig_mask
                                & (all_params[...,k][...,0]>SIG_THRESHOLD)).sum())
    p   = all_params[..., best_k]
    ok  = (p[..., 5] > 0.5) & sig_mask & (p[..., 0] > SIG_THRESHOLD)
    centres_deg  = p[ok, 2]
    fwhm_deg     = np.abs(p[ok, 1]) * FWHM
    amps         = p[ok, 0]

    print(f'Strain sweep — best µ layer: k={best_k}  µ={mu_values[best_k]:.3f}°')
    print(f'  Grain pixels with good fit : {ok.sum():,}')
    print(f'  ccmth peak centre          : {np.median(centres_deg):.5f}°  '
          f'±{np.std(centres_deg)*1000:.2f} mdeg spread')
    print(f'  Median FWHM                : {np.median(fwhm_deg)*1000:.2f} mdeg')
    print(f'  Median amplitude           : {np.median(amps):.0f} counts')

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    dev = (centres_deg - np.median(centres_deg)) * 1000
    axes[0].hist(dev, bins=80, color='steelblue', edgecolor='none', alpha=0.85)
    axes[0].axvline(0, color='k', lw=1.5, ls='--')
    axes[0].set_xlabel('Δccmth from median (mdeg)')
    axes[0].set_ylabel('Grain pixels')
    axes[0].set_title(f'Strain distribution — µ={mu_values[best_k]:.3f}°')
    axes[0].grid(True, alpha=0.2)

    axes[1].hist(fwhm_deg * 1000, bins=80, color='coral', edgecolor='none', alpha=0.85)
    axes[1].set_xlabel('FWHM (mdeg)')
    axes[1].set_ylabel('Grain pixels')
    axes[1].set_title('Rocking width distribution')
    axes[1].grid(True, alpha=0.2)

    axes[2].plot(mu_values,
                 [(all_params[...,k][...,0][sig_mask].mean()
                   if sig_mask.any() else 0) for k in range(n_mu)],
                 'o-', color='seagreen', lw=2)
    axes[2].set_xlabel('µ (°)'); axes[2].set_ylabel('Mean amplitude (cts)')
    axes[2].set_title('Signal strength vs depth'); axes[2].grid(True, alpha=0.2)

    plt.tight_layout(); plt.show()


elif SCAN_TYPE == 'mosa':
    ok = (p2d[..., 7] > 0.5) & sig_mask
    chi_px   = p2d[ok, 1]
    mu_px    = p2d[ok, 2]
    fchi_px  = FWHM * np.sqrt(np.maximum(p2d[ok, 3], 0)) * 1000   # mdeg
    fmu_px   = FWHM * np.sqrt(np.maximum(p2d[ok, 5], 0)) * 1000   # mdeg

    print('Mosaicity statistics:')
    print(f'  Grain pixels with good fit : {ok.sum():,}')
    print(f'  chi orientation centre     : {np.median(chi_px):.4f}°  '
          f'±{np.std(chi_px)*1000:.2f} mdeg spread')
    print(f'  µ orientation centre       : {np.median(mu_px):.4f}°  '
          f'±{np.std(mu_px)*1000:.2f} mdeg spread')
    print(f'  Median FWHM chi            : {np.median(fchi_px):.1f} mdeg')
    print(f'  Median FWHM µ              : {np.median(fmu_px):.1f} mdeg')
    print(f'  Median total mosaicity     : '
          f'{np.median(np.sqrt(fchi_px**2 + fmu_px**2)):.1f} mdeg')

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].hist((chi_px - np.median(chi_px))*1000, bins=80,
                 color='steelblue', edgecolor='none', alpha=0.85)
    axes[0].axvline(0, color='k', lw=1.5, ls='--')
    axes[0].set_xlabel('Δchi from median (mdeg)'); axes[0].set_ylabel('Grain pixels')
    axes[0].set_title('chi orientation spread'); axes[0].grid(True, alpha=0.2)

    axes[1].hist((mu_px - np.median(mu_px))*1000, bins=80,
                 color='coral', edgecolor='none', alpha=0.85)
    axes[1].axvline(0, color='k', lw=1.5, ls='--')
    axes[1].set_xlabel('Δµ from median (mdeg)')
    axes[1].set_title('µ orientation spread'); axes[1].grid(True, alpha=0.2)

    axes[2].hist(fchi_px, bins=80, color='seagreen', edgecolor='none',
                 alpha=0.7, label='chi')
    axes[2].hist(fmu_px, bins=80, color='purple', edgecolor='none',
                 alpha=0.5, label='µ')
    axes[2].set_xlabel('FWHM (mdeg)')
    axes[2].set_title('Rocking-curve width distributions'); axes[2].legend()
    axes[2].grid(True, alpha=0.2)

    plt.tight_layout(); plt.show()


elif SCAN_TYPE == 'strain_mosa_3d':
    print('3D strain-mosa statistics:')
    print(f'  Grain pixels (>{SIG_THRESHOLD} cts)  : {sig_mask.sum():,}')
    for i, lbl in enumerate(['chi', 'µ', 'obpitch']):
        v = mean_3d[sig_mask, i]
        print(f'  {lbl} COM : {np.median(v):.4f}°  ±{np.std(v)*1000:.2f} mdeg spread')
    print(f'  Peak intensity layer : obpitch={ob_values[peak_layer]:.4f}° '
          f'(layer {peak_layer+1}/{n_ob})')

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    labels_3d  = ['chi COM', 'µ COM', 'obpitch COM']
    colors_3d  = ['steelblue', 'coral', 'seagreen']
    for i in range(3):
        v = mean_3d[sig_mask, i]
        axes[i].hist((v - np.median(v))*1000, bins=80,
                     color=colors_3d[i], edgecolor='none', alpha=0.85)
        axes[i].axvline(0, color='k', lw=1.5, ls='--')
        axes[i].set_xlabel(f'Δ{labels_3d[i]} from median (mdeg)')
        axes[i].set_ylabel('Grain pixels')
        axes[i].set_title(f'{labels_3d[i]} distribution')
        axes[i].grid(True, alpha=0.2)

    plt.tight_layout(); plt.show()

## 11 · Save results

In [ ]:
OUT = f'/Users/matt/Lab/projects/DFXM/analysis/notebook_{SCAN_TYPE}'
os.makedirs(OUT, exist_ok=True)

np.save(f'{OUT}/sig_mask.npy', sig_mask)
np.save(f'{OUT}/motors.npy',   dset.motors)

if SCAN_TYPE == 'strain_sweep':
    np.save(f'{OUT}/params.npy', all_params)   # (ny, nx, 6, n_mu)
elif SCAN_TYPE == 'mosa':
    np.save(f'{OUT}/gauss2d.npy',      p2d)
    np.save(f'{OUT}/moments_mean.npy', mean_map)
    np.save(f'{OUT}/moments_cov.npy',  cov_map)
elif SCAN_TYPE == 'strain_mosa_3d':
    np.save(f'{OUT}/moments_mean_3d.npy', mean_3d)
    np.save(f'{OUT}/moments_cov_3d.npy',  cov_3d)

# Optional: save a full-resolution result map as a PNG for your presentation
# fig.savefig(f'{OUT}/result_maps.png', dpi=200, bbox_inches='tight')

print(f'Saved to {OUT}/')
for f in sorted(os.listdir(OUT)):
    sz = os.path.getsize(f'{OUT}/{f}') / 1e6
    print(f'  {f:<35}  {sz:.1f} MB')